# 실습 17. 경쟁위험 분석

사회약학 연구방법 노트의 '실습 17 경쟁위험 분석'에 나오는 셀을 순서대로 담은 노트북입니다. 17장의 예제 자료(가상의 만성콩팥병 코호트, 약물 A 대 B)로 누적발생함수, 원인별 위험비, Fine–Gray 하위분포 위험비를 구합니다. 셀을 위에서부터 차례로 실행하세요. 셀 18은 줄 수가 많아 10초 넘게 걸릴 수 있습니다. 설명은 사이트의 실습 17 쪽에 있습니다.


## 가. 실습 데이터 준비

### 셀 1. lifelines 설치

In [ ]:
!pip install lifelines

### 셀 2. 자료 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
UA = {"User-Agent": "Mozilla/5.0"}   # 사이트가 파이썬 기본 요청을 막아 브라우저처럼 보이게 함
ckd = pd.read_csv(BASE + "ckd_competing.csv", storage_options=UA)
print(ckd.shape)
ckd.head()

### 셀 3. 사건 코딩 확인과 사건 열 만들기

In [ ]:
ckd["drug_a"] = (ckd["drug"] == "A").astype(int)   # A = 1, B = 0
ckd["dial"] = (ckd["status"] == 1).astype(int)     # 투석 시작
ckd["death"] = (ckd["status"] == 2).astype(int)    # 투석 전 사망

tab = pd.crosstab(ckd["status"], ckd["drug"], margins=True)
tab.index = ["0 censored", "1 dialysis", "2 death", "All"]
tab

### 셀 4. 중도절단이 자료 마감뿐인지 확인

In [ ]:
cens = ckd["status"] == 0
same = ckd.loc[cens, "years"] == ckd.loc[cens, "max_years"]
early = ckd.loc[~cens, "years"] < ckd.loc[~cens, "max_years"]
print("중도절단 시점 = 자료 마감:", same.all())
print("사건은 자료 마감 전     :", early.all())
ckd.groupby("drug")["max_years"].describe().round(2)

### 셀 5. 군별 사건 수, 인년, 발생률

In [ ]:
rate = ckd.groupby("drug").agg(
    n=("id", "size"), py=("years", "sum"),
    dial=("dial", "sum"), death=("death", "sum"))
rate["mean_fu"] = rate["py"] / rate["n"]
rate["dial_100py"] = rate["dial"] / rate["py"] * 100
rate["death_100py"] = rate["death"] / rate["py"] * 100
print(rate.round(2).T)

ratio = rate.loc["A"] / rate.loc["B"]
print("A / B:",
      ratio[["dial_100py", "death_100py"]].round(2).tolist())

## 나. 누적발생함수

### 셀 6. 약물 B군의 누적발생률과 1 − Kaplan-Meier

In [ ]:
from lifelines import AalenJohansenFitter, KaplanMeierFitter

b = ckd[ckd["drug"] == "B"]
ajf = AalenJohansenFitter()
ajf.fit(b["years"], b["status"], event_of_interest=1)
print(ajf.predict([1, 3, 5]))
print(ajf.confidence_interval_.loc[:5].tail(1))

kmf = KaplanMeierFitter()
kmf.fit(b["years"], b["status"] == 1)      # 사망도 중도절단이 됨
print("1 - KM:", round(1 - kmf.predict(5), 4))

### 셀 7. 두 군의 1·3·5년 투석 누적발생률 (95% 신뢰구간)

In [ ]:
def cif(d, event, t):
    """t년 누적발생률, 95% 신뢰구간의 하한과 상한 (0-1)"""
    aj = AalenJohansenFitter()
    aj.fit(d["years"], d["status"], event_of_interest=event)
    lo, hi = sorted(aj.confidence_interval_.loc[:t].iloc[-1])
    return aj.predict(t), lo, hi

rows = {}
for g in ["A", "B"]:
    d = ckd[ckd["drug"] == g]
    for t in [1, 3, 5]:
        est, lo, hi = cif(d, 1, t)
        rows[(f"{t}-year", g)] = (f"{est*100:.1f} "
                                  f"({lo*100:.1f}-{hi*100:.1f})")
pd.Series(rows).unstack()

### 셀 8. 5년 시점의 세 상태와 합계, 1 − Kaplan-Meier

In [ ]:
rows = []
for g in ["A", "B"]:
    d = ckd[ckd["drug"] == g]
    km_any = KaplanMeierFitter().fit(d["years"], d["status"] > 0)
    km_dial = KaplanMeierFitter().fit(d["years"], d["status"] == 1)
    rows.append({"drug": g,
                 "dialysis": cif(d, 1, 5)[0],
                 "death": cif(d, 2, 5)[0],
                 "event_free": km_any.predict(5),
                 "km_dialysis": 1 - km_dial.predict(5)})
five = pd.DataFrame(rows).set_index("drug") * 100
five["sum"] = five["dialysis"] + five["death"] + five["event_free"]
five.round(1)

### 셀 9. 누적발생함수와 1 − Kaplan-Meier 겹쳐 그리기

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, g in zip(axes, ["A", "B"]):
    d = ckd[ckd["drug"] == g]
    aj = AalenJohansenFitter()
    aj.fit(d["years"], d["status"], event_of_interest=1)
    km = KaplanMeierFitter().fit(d["years"], d["status"] == 1)
    one_minus_km = 1 - km.survival_function_["KM_estimate"]
    ax.step(one_minus_km.index, one_minus_km, where="post",
            ls="--", label="1 - KM (death censored)")
    ax.step(aj.cumulative_density_.index,
            aj.cumulative_density_["CIF_1"], where="post",
            label="Cumulative incidence (AJ)")
    ax.set_title(f"Drug {g}")
    ax.set_xlabel("Years since first prescription")
axes[0].set_ylabel("Proportion starting dialysis")
axes[0].legend()
plt.tight_layout()

### 셀 10. 세 상태를 쌓아 그리기

In [ ]:
grid = np.linspace(0, 5, 251)                # 0, 0.02, ..., 5년
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, g in zip(axes, ["A", "B"]):
    d = ckd[ckd["drug"] == g]
    f1 = AalenJohansenFitter().fit(
        d["years"], d["status"], event_of_interest=1).predict(grid)
    f2 = AalenJohansenFitter().fit(
        d["years"], d["status"], event_of_interest=2).predict(grid)
    ax.stackplot(grid, f1, f2, 1 - f1 - f2,
                 labels=["Dialysis", "Death before dialysis",
                         "Alive without dialysis"])
    ax.set_title(f"Drug {g}")
    ax.set_xlabel("Years since first prescription")
    ax.set_xlim(0, 5)
axes[0].set_ylabel("Proportion of patients")
axes[0].set_ylim(0, 1)
axes[0].legend(loc="upper right")
plt.tight_layout()

### 셀 11. 5년 투석 누적발생률의 차이와 95% 신뢰구간

In [ ]:
est = {}
for g in ["A", "B"]:
    d = ckd[ckd["drug"] == g]
    aj = AalenJohansenFitter()
    aj.fit(d["years"], d["status"], event_of_interest=1)
    est[g] = (aj.predict(5), aj.variance_.loc[:5].iloc[-1])
(fa, va), (fb, vb) = est["A"], est["B"]

rd = fa - fb                                 # 위험차
se = np.sqrt(va + vb)
print(f"difference {rd*100:.1f}%p (95% CI {(rd - 1.96*se)*100:.1f}"
      f" to {(rd + 1.96*se)*100:.1f})")
print("ratio", round(fa / fb, 2))

### 셀 12. 일 단위 자료에서 나오는 동점 경고

In [ ]:
from statsmodels.duration.survfunc import CumIncidenceRight

days = np.ceil(b["years"] * 365.25)          # 일 단위로 바꿈
aj_d = AalenJohansenFitter(seed=1)
aj_d.fit(days, b["status"], event_of_interest=1)
print("lifelines (jitter) :", round(aj_d.predict(1827), 4))

cir = CumIncidenceRight(days, b["status"])
print("statsmodels        :", round(cir.cinc[0][-1], 4))

## 다. 원인별 위험비와 Fine–Gray 위험비

### 셀 13. 원인별 Cox 모형 (투석, 투석 전 사망)

In [ ]:
from lifelines import CoxPHFitter

cols = ["exp(coef)", "exp(coef) lower 95%",
        "exp(coef) upper 95%", "p"]
cs_dial = CoxPHFitter().fit(ckd, "years", "dial", formula="drug_a")
cs_death = CoxPHFitter().fit(ckd, "years", "death", formula="drug_a")
print(cs_dial.summary[cols].round(3))
print(cs_death.summary[cols].round(3))

### 셀 14. 3년 시점 약물 B군의 두 가지 위험집합

In [ ]:
t0 = 3
at_risk = b["years"] >= t0                 # 3년까지 사건 없이 추적 중
died_before = ((b["status"] == 2) & (b["years"] < t0)
               & (b["max_years"] >= t0))   # 3년 전에 사망
print("cause-specific risk set :", at_risk.sum())
print("died before, kept       :", died_before.sum())
print("subdistribution risk set:", (at_risk | died_before).sum())

### 셀 15. Fine–Gray 모형 (모든 환자의 자료 마감 시점을 알 때)

In [ ]:
died = ckd["status"] == 2
# 투석 전 사망자의 추적 시간을 자료 마감까지로 늘림
ckd["years_sd"] = np.where(died, ckd["max_years"], ckd["years"])

fg = CoxPHFitter().fit(ckd, "years_sd", "dial", formula="drug_a")
print(fg.summary[cols].round(3))
print("p =", round(fg.summary.loc["drug_a", "p"], 4))

### 셀 16. 중도절단되지 않고 남을 확률 G(t)

In [ ]:
# 중도절단을 '사건'으로 놓은 Kaplan-Meier = 추적이 이어질 확률 G(t)
kmc = KaplanMeierFitter().fit(ckd["years"], ckd["status"] == 0)
print(kmc.predict([1, 3, 3.5, 4, 4.5, 4.99]).round(3))

### 셀 17. 사망자를 위험집합에 남긴 긴 자료와 가중치

In [ ]:
ev_t = np.sort(ckd.loc[ckd["dial"] == 1, "years"].unique())
dead = ckd.loc[died, ["id", "drug_a", "years"]]

# 사망자마다, 사망 뒤에 투석이 일어난 시점들을 한 줄씩 붙임
ext = dead.merge(pd.DataFrame({"stop": ev_t}), how="cross")
ext = ext[ext["stop"] > ext["years"]].copy()
ext["w"] = (kmc.predict(ext["stop"]).values
            / kmc.predict(ext["years"]).values)
# 가중치가 같은 이웃 줄은 마지막 줄만 남겨 한 줄로 합침
nxt = ext.groupby("id")["w"].shift(-1)     # 같은 환자의 다음 줄 w
ext = ext[ext["w"] != nxt].copy()
prev = ext.groupby("id")["stop"].shift()   # 같은 환자의 앞 줄 stop
ext["start"] = prev.fillna(ext["years"])   # 첫 줄은 사망 시점부터
ext["dial"] = 0

own = ckd.assign(start=0.0, stop=ckd["years"], w=1.0)
keep = ["id", "drug_a", "start", "stop", "dial", "w"]
long = pd.concat([own[keep], ext[keep]], ignore_index=True)
print(len(ckd), "명 ->", len(long), "줄")
long[long["id"] == 1].round(4)

### 셀 18. Fine–Gray 모형 (가중치를 쓰는 일반적인 방법)

In [ ]:
from lifelines import CoxTimeVaryingFitter

fgw = CoxTimeVaryingFitter()
fgw.fit(long, id_col="id", event_col="dial", start_col="start",
        stop_col="stop", weights_col="w", formula="drug_a")
print(fgw.summary[cols].round(3))

### 셀 19. 누적발생함수의 비교 (Gray 검정의 원리)와 보통의 로그순위 검정

In [ ]:
from lifelines.statistics import logrank_test

a = ckd["drug_a"] == 1
gray = logrank_test(ckd.loc[a, "years_sd"], ckd.loc[~a, "years_sd"],
                    ckd.loc[a, "dial"], ckd.loc[~a, "dial"])
lr = logrank_test(ckd.loc[a, "years"], ckd.loc[~a, "years"],
                  ckd.loc[a, "dial"], ckd.loc[~a, "dial"])
print(f"deaths kept in risk set: chi2 {gray.test_statistic:.2f},"
      f" P {gray.p_value:.4f}")
print(f"deaths censored        : chi2 {lr.test_statistic:.2f},"
      f" P {lr.p_value:.4f}")

### 셀 20. 위험비를 한 표로 정리

In [ ]:
def hr_ci(m, var="drug_a"):
    s = m.summary.loc[var]
    return (f"{s['exp(coef)']:.2f} ({s['exp(coef) lower 95%']:.2f}"
            f"-{s['exp(coef) upper 95%']:.2f})")

pd.DataFrame({"HR (95% CI), A vs B": {
    "dialysis: cause-specific": hr_ci(cs_dial),
    "dialysis: subdistribution": hr_ci(fg),
    "dialysis: subdistribution (weighted)": hr_ci(fgw),
    "death: cause-specific": hr_ci(cs_death)}})

## 과제 정답

**과제 1. 투석 전 사망을 관심 사건으로.** 두 군의 5년 사망 누적발생률(95% 신뢰구간)과, 투석을 중도절단으로 둔 1 − Kaplan-Meier를 구합니다. 이어서 사망의 하위분포 위험비를 구해 셀 13의 원인별 위험비와 비교합니다.

### 셀 21. 과제 1 정답. 투석 전 사망을 관심 사건으로

In [ ]:
for g in ["A", "B"]:
    d = ckd[ckd["drug"] == g]
    est, lo, hi = cif(d, 2, 5)             # 관심 사건 = 사망(2)
    km = KaplanMeierFitter().fit(d["years"], d["status"] == 2)
    print(f"{g}: CIF {est*100:.1f} ({lo*100:.1f}-{hi*100:.1f}),"
          f" 1 - KM {(1 - km.predict(5))*100:.1f}")

# 이번에는 투석을 시작한 사람을 자료 마감까지 위험집합에 남김
dialyzed = ckd["status"] == 1
ckd["years_sd2"] = np.where(dialyzed, ckd["max_years"], ckd["years"])
fg_death = CoxPHFitter().fit(ckd, "years_sd2", "death",
                             formula="drug_a")
print("subdistribution HR:", hr_ci(fg_death))
print("cause-specific HR :", hr_ci(cs_death))

**과제 2. 복합 결과.** '투석 또는 사망 중 먼저 일어난 것'을 사건으로 놓고 Cox 모형의 위험비와 두 군의 5년 1 − Kaplan-Meier를 구합니다. 이 값이 투석과 사망의 누적발생률의 합과 같은지 확인합니다.

### 셀 22. 과제 2 정답. 복합 결과(투석 또는 사망)

In [ ]:
ckd["any"] = (ckd["status"] > 0).astype(int)   # 투석 또는 사망
comp = CoxPHFitter().fit(ckd, "years", "any", formula="drug_a")
print("composite HR:", hr_ci(comp))

for g in ["A", "B"]:
    d = ckd[ckd["drug"] == g]
    km = KaplanMeierFitter().fit(d["years"], d["any"])
    both = cif(d, 1, 5)[0] + cif(d, 2, 5)[0]
    print(f"{g}: 1 - KM {(1 - km.predict(5))*100:.1f},"
          f" CIF dialysis + CIF death {both*100:.1f}")

**과제 3. 고관절 골절 자료.** dementia_fracture.csv(치매 고령 환자, 약물 C 대 D, 1 = 고관절 골절, 2 = 골절 전 사망)로 5년 누적발생률, 1 − Kaplan-Meier, 골절과 사망의 원인별 위험비, 골절의 하위분포 위험비를 구합니다.

### 셀 23. 과제 3 정답. 고관절 골절 자료 (17장 나 절 확인 문제)

In [ ]:
fx = pd.read_csv(BASE + "dementia_fracture.csv", storage_options=UA)
fx["drug_c"] = (fx["drug"] == "C").astype(int)     # C = 1, D = 0
fx["frac"] = (fx["status"] == 1).astype(int)
fx["death"] = (fx["status"] == 2).astype(int)
fx["years_sd"] = np.where(fx["status"] == 2,
                          fx["max_years"], fx["years"])

for g in ["C", "D"]:
    d = fx[fx["drug"] == g]
    est, lo, hi = cif(d, 1, 5)
    km = KaplanMeierFitter().fit(d["years"], d["status"] == 1)
    print(f"{g}: fracture {est*100:.1f} ({lo*100:.1f}-{hi*100:.1f}),"
          f" death {cif(d, 2, 5)[0]*100:.1f},"
          f" 1 - KM {(1 - km.predict(5))*100:.1f}")

m1 = CoxPHFitter().fit(fx, "years", "frac", formula="drug_c")
m2 = CoxPHFitter().fit(fx, "years", "death", formula="drug_c")
m3 = CoxPHFitter().fit(fx, "years_sd", "frac", formula="drug_c")
print("fracture, cause-specific HR :", hr_ci(m1, "drug_c"))
print("death, cause-specific HR    :", hr_ci(m2, "drug_c"))
print("fracture, subdistribution HR:", hr_ci(m3, "drug_c"))